In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf


df = yf.download("GC=F", start="2026-01-01", end="2026-05-30")
print(df.head())

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='gold', linewidth=2)
plt.title("Gold Price - 2026")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
war_events = {
    "2026-02-28": "War Starts",
    "2026-03-09": "Hormuz Threat",
    "2026-04-08": "Ceasefire"
}

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], color='gold', linewidth=2)

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date),
                color='red', linestyle='--', alpha=0.8)
    plt.text(pd.to_datetime(date),
             df['Close'].max() * 0.98,
             label, rotation=45, fontsize=9, color='red')

plt.title("Gold Price During 2026 Iran War")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
pre_war = df.loc['2026-02-01':'2026-02-27', ('Close', 'GC=F')].mean()
peak = df.loc['2026-03-01':'2026-03-10', ('Close', 'GC=F')].max()
spike = ((peak - pre_war) / pre_war) * 100

print(f"Pre-war avg gold price: ${pre_war:.2f}")
print(f"Peak gold price: ${peak:.2f}")
print(f"Gold spiked: {spike:.1f}% in first 10 days of war")

In [ ]:
# Download oil alongside gold
oil = yf.download("CL=F", start="2026-01-01", end="2026-05-30")
sp500 = yf.download("^GSPC", start="2026-01-01", end="2026-05-30")
gld_etf = yf.download("GLD", start="2026-01-01", end="2026-05-30")  # Gold ETF — this shows investor flows
vix = yf.download("^VIX", start="2026-01-01", end="2026-05-30")     # Fear index

print("Oil data:", oil.shape)
print("SP500 data:", sp500.shape)

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(oil.index, oil['Close'], color='black', linewidth=2)

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date), color='red', linestyle='--', alpha=0.8)
    plt.text(pd.to_datetime(date), oil['Close'].max()*0.98,
             label, rotation=45, fontsize=9, color='red')

plt.title("WTI Oil Price During 2026 Iran War")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Combining everything into one dataframe
import pandas as pd

flows = pd.DataFrame({
    'Gold_Price': df[('Close', 'GC=F')],
    'Oil_Price': oil[('Close', 'CL=F')],
    'SP500_Price': sp500[('Close', '^GSPC')],
    'Gold_ETF_Volume': gld_etf[('Volume', 'GLD')],  # investors rushing INTO gold
    'SP500_Volume': sp500[('Volume', '^GSPC')],         # investors rushing OUT of stocks
    'Fear_VIX': vix[('Close', '^VIX')],
})


# This shows % change from starting point
base = flows['2026-01-01':'2026-02-27']
flows_normalized = (flows / flows.iloc[0]) * 100

# Plotting all assets together
plt.figure(figsize=(16, 7))
plt.plot(flows_normalized['Gold_Price'], label='Gold', color='gold', linewidth=2)
plt.plot(flows_normalized['Oil_Price'], label='Oil', color='black', linewidth=2)
plt.plot(flows_normalized['SP500_Price'], label='S&P 500', color='blue', linewidth=2)
plt.plot(flows_normalized['Fear_VIX'], label='Fear Index (VIX)',
         color='red', linewidth=2, linestyle='--')

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date), color='red', linestyle=':', alpha=0.6)

plt.axhline(100, color='gray', linestyle='-', alpha=0.3)  # baseline
plt.title("Capital Flows — Where Did Investor Money Go During 2026 Iran War?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# How much did SP500 lose vs how much gold gained
pre_war_date = '2026-02-27'
peak_war_date = '2026-03-10'

sp500_change = ((sp500[('Close', '^GSPC')][peak_war_date] - sp500[('Close', '^GSPC')][pre_war_date])
                / sp500[('Close', '^GSPC')][pre_war_date] * 100)

gold_change = ((df[('Close', 'GC=F')][peak_war_date] - df[('Close', 'GC=F')][pre_war_date])
               / df[('Close', 'GC=F')][pre_war_date] * 100)

oil_change = ((oil[('Close', 'CL=F')][peak_war_date] - oil[('Close', 'CL=F')][pre_war_date])
              / oil[('Close', 'CL=F')][pre_war_date] * 100)

print(f"S&P 500 changed: {sp500_change:.1f}%")
print(f"Gold changed: {gold_change:.1f}%")
print(f"Oil changed: {oil_change:.1f}%")

In [ ]:
import seaborn as sns

corr = flows[['Gold_Price', 'Oil_Price',
              'SP500_Price', 'Fear_VIX']].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm',
            center=0, fmt='.2f')
plt.title("Asset Correlation During 2026 Iran War")
plt.show()

In [ ]:
pre_war = flows['2026-01-01':'2026-02-27']
during_war = flows['2026-02-28':'2026-04-08']
ceasefire = flows['2026-04-09':'2026-05-30']

vol_table = pd.DataFrame({
    'Pre War': pre_war[['Gold_Price','Oil_Price',
                        'SP500_Price','Fear_VIX']].std(),
    'During War': during_war[['Gold_Price','Oil_Price',
                              'SP500_Price','Fear_VIX']].std(),
    'Ceasefire': ceasefire[['Gold_Price','Oil_Price',
                            'SP500_Price','Fear_VIX']].std()
})

print(vol_table)

In [ ]:
lmt = yf.download("LMT", start="2026-01-01", end="2026-05-30")
rtx = yf.download("RTX", start="2026-01-01", end="2026-05-30")

In [ ]:
# Normalizing everything to 100 for fair comparison
defense = pd.DataFrame({
    'Lockheed_Martin': (lmt[('Close', 'LMT')] / lmt[('Close', 'LMT')].iloc[0]) * 100,
    'Raytheon': (rtx[('Close', 'RTX')] / rtx[('Close', 'RTX')].iloc[0]) * 100,
    'SP500': (sp500[('Close', '^GSPC')] / sp500[('Close', '^GSPC')].iloc[0]) * 100,
})

# Plotting
plt.figure(figsize=(14,6))
plt.plot(defense['Lockheed_Martin'],
         label='Lockheed Martin (LMT)', color='green', linewidth=2)
plt.plot(defense['Raytheon'],
         label='Raytheon (RTX)', color='blue', linewidth=2)
plt.plot(defense['SP500'],
         label='S&P 500', color='red', linewidth=2, linestyle='--')

for date, label in war_events.items():
    plt.axvline(pd.to_datetime(date),
                color='red', linestyle=':', alpha=0.6)
    plt.text(pd.to_datetime(date),
             defense.max().max() * 0.98,
             label, rotation=45, fontsize=8, color='red')

plt.axhline(100, color='gray', linestyle='-', alpha=0.3)
plt.title("Defense Stocks vs S&P 500 — Who Made Money During War?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Printing exact numbers
pre = '2026-02-27'
peak = '2026-03-10'

lmt_change = ((lmt[('Close', 'LMT')][peak] - lmt[('Close', 'LMT')][pre]) / lmt[('Close', 'LMT')][pre] * 100)
rtx_change = ((rtx[('Close', 'RTX')][peak] - rtx[('Close', 'RTX')][pre]) / rtx[('Close', 'RTX')][pre] * 100)
sp_change = ((sp500[('Close', '^GSPC')][peak] - sp500[('Close', '^GSPC')][pre]) / sp500[('Close', '^GSPC')][pre] * 100)

print(f"Lockheed Martin: {float(lmt_change):.1f}%")
print(f"Raytheon: {float(rtx_change):.1f}%")
print(f"S&P 500: {float(sp_change):.1f}%")